In questo notebook studiamo 2 metodi di ottimizzazione. 

1. **Gradient Descent Classico**: La base di partenza, con i suoi limiti strutturali.
2. **Gradient Descent con Momentum**: L'introduzione della fisica (velocità e inerzia) per risolvere le oscillazioni.

Implementeremo ogni metodo da zero in Python (con NumPy) su una funzione bidimensionale "a valle" per visualizzare graficamente le traiettorie e capire visivamente le differenze.

## 1. Il Gradient Descent Classico (Discesa del Gradiente)

Il Gradient Descent è l'algoritmo di ottimizzazione più semplice. Il suo obiettivo è trovare il minimo di una funzione di costo $f(w)$ muovendosi nella direzione opposta a quella indicata dal gradiente (vettore che indica la direzione di massima crescita della funzione).

### La Formula Matematica
$$w_{t+1} = w_t - \eta \cdot \nabla f(w_t)$$

Dove:
*   $w_t$ sono i parametri (pesi) al passo $t$.
*   $\eta$ (eta) è il **Learning Rate** (tasso di apprendimento), ovvero la lunghezza del passo.
*   $\nabla f(w_t)$ è il **gradiente** (il vettore delle derivate parziali) che indica la direzione di massima salita.

### Il Limite (Il problema delle "Valli Strette"):
Immagina una valle molto stretta e allungata. La pendenza lungo i fianchi della valle (asse $y$) è molto ripida, mentre la pendenza che porta al fondo della valle verso il minimo (asse $x$) è molto dolce. 
Il Gradient Descent classico finirà per fare passi enormi rimbalzando da un fianco all'altro dell'asse $y$, avanzando pochissimo e molto lentamente lungo l'asse $x$.

## 2. Gradient Descent con Momentum (La palla che rotola)

Per risolvere il problema dei continui rimbalzi e accelerare la discesa, si richiama un concetto dalla fisica classica: **l'inerzia (o quantità di moto)**.

Invece di decidere la direzione basandoci *solo* sul gradiente del passo corrente, ricordiamo in quale direzione ci stavamo muovendo nei passi precedenti. 
L'algoritmo si comporta come una palla pesante che rotola lungo una discesa: accumula velocità lungo le direzioni costanti e smorza le oscillazioni.

### Le Formule Matematiche
1. **Aggiornamento della velocità ($v$):**
   $$v_{t+1} = \beta \cdot v_t + \nabla f(w_t)$$
2. **Aggiornamento dei parametri ($w$):**
   $$w_{t+1} = w_t - \eta \cdot v_{t+1}$$

Dove:
*   $v_t$ è la velocità accumulata fino al passo precedente.
*   $\beta$ (beta) è il coefficiente di attrito/momentum (solitamente impostato tra $0.9$ e $0.99$). Determina quanta importanza dare alla velocità passata rispetto al nuovo gradiente.

### Perché funziona?
Lungo l'asse delle oscillazioni ($y$), i gradienti cambiano continuamente segno (una volta positivi, una volta negativi) e si annullano a vicenda nella media della velocità $v$. 
Lungo l'asse della discesa ($x$), i gradienti spingono sempre nella stessa direzione; quindi la velocità in quella direzione si somma e l'algoritmo accelera verso il minimo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definiamo una Loss Function bidimensionale "a valle stretta"
# f(x, y) = 0.1 * x^2 + 2 * y^2
def loss_2d(x, y):
    return 0.1 * (x ** 2) + 2.0 * (y ** 2)

# Calcolo del gradiente analitico per x e y
def grad_2d(x, y):
    return 0.2 * x, 4.0 * y

# 1. IMPLEMENTAZIONE GRADIENT DESCENT CLASSICO
def run_gd(start_x, start_y, lr, epochs):
    x, y = start_x, start_y
    path = [(x, y)]
    for _ in range(epochs):
        gx, gy = grad_2d(x, y)
        x = x - lr * gx
        y = y - lr * gy
        path.append((x, y))
    return np.array(path)

# 2. IMPLEMENTAZIONE MOMENTUM
def run_momentum(start_x, start_y, lr, beta, epochs):
    x, y = start_x, start_y
    vx, vy = 0.0, 0.0  # Velocità iniziali
    path = [(x, y)]
    for _ in range(epochs):
        gx, gy = grad_2d(x, y)
        vx = beta * vx + gx
        vy = beta * vy + gy
        x = x - lr * vx
        y = y - lr * vy
        path.append((x, y))
    return np.array(path)

# --- CONFIGURAZIONE TEST ---
start_x, start_y = 8.0, 1.5
epochs = 40
lr = 0.4

# Esecuzione degli algoritmi
path_gd = run_gd(start_x, start_y, lr=0.3, epochs=epochs) # lr ridotto per evitare divergenza sul GD classico
path_mom = run_momentum(start_x, start_y, lr=0.15, beta=0.9, epochs=epochs)

# --- CREAZIONE DEL GRAFICO ---
x_vals = np.linspace(-10, 10, 100)
y_vals = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x_vals, y_vals)
Z = loss_2d(X, Y)

plt.figure(figsize=(14, 8))
plt.contour(X, Y, Z, levels=25, cmap='viridis', alpha=0.6)

# Tracciamento dei percorsi
plt.plot(path_gd[:, 0], path_gd[:, 1], '-o', color='red', label='GD Classico (Oscillante)', markersize=4, alpha=0.8)
plt.plot(path_mom[:, 0], path_mom[:, 1], '-s', color='blue', label='Momentum (Direzionale)', markersize=4, alpha=0.8)

plt.plot(0, 0, '*', color='gold', markersize=18, label='Minimo Globale', zorder=10)
plt.title("Confronto Traiettorie: Gradient Descent vs Momentum", fontsize=14)
plt.xlabel("Parametro X (Pendenza Dolce)", fontsize=12)
plt.ylabel("Parametro Y (Pendenza Ripida)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Come usarli in PyTorch

Ora che abbiamo implementato la matematica da zero per capire il "dietro le quinte", passiamo al mondo reale. Nelle applicazioni pratiche di Deep Learning non riscriviamo gli ottimizzatori da zero, ma utilizziamo le implementazioni ottimizzate offerte da **PyTorch** nel modulo `torch.optim`.

In [ ]:
import torch
import torch.optim as optim

# Parametro iniziale
initial_value = 10.0

print("--- Esempio in PyTorch ---")

# 1. Stochastic Gradient Descent (SGD) classico
w_gd = torch.tensor([initial_value], requires_grad=True)
optimizer_gd = optim.SGD([w_gd], lr=0.1)

# 2. SGD con Momentum abilitato
w_mom = torch.tensor([initial_value], requires_grad=True)
optimizer_mom = optim.SGD([w_mom], lr=0.1, momentum=0.9)

# 3. ADAM
w_adam = torch.tensor([initial_value], requires_grad=True)
optimizer_adam = optim.Adam([w_adam], lr=0.1)

# Simuliamo 3 epoche di addestramento su una funzione di loss f(w) = w^2
for epoch in range(1, 4):
    print(f"\n[Epoca {epoch}]")
    
    # Calcolo Loss e Step per GD
    optimizer_gd.zero_grad()
    loss_gd = w_gd ** 2
    loss_gd.backward()
    optimizer_gd.step()
    
    # Calcolo Loss e Step per Momentum
    optimizer_mom.zero_grad()
    loss_mom = w_mom ** 2
    loss_mom.backward()
    optimizer_mom.step()
    
    # Calcolo Loss e Step per ADAM
    optimizer_adam.zero_grad()
    loss_adam = w_adam ** 2
    loss_adam.backward()
    optimizer_adam.step()
    
    print(f" GD Classico -> Parametro w: {w_gd.item():.4f}")
    print(f" SGD Momentum -> Parametro w: {w_mom.item():.4f}")
    print(f" ADAM        -> Parametro w: {w_adam.item():.4f}")